In [23]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub
import tensorly as tl
import plotly.io as pio
pio.renderers.default = 'iframe'
tl.set_backend('cupy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')
print(f'TensorLy tenalg backend: {tl.tenalg.get_backend()}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: CUPY_ACCELERATORS=cutensor,cub
TensorLy backend: cupy
TensorLy tenalg backend: einsum


In [24]:
from moabb.paradigms import FilterBankMotorImagery, MotorImagery
from moabb.datasets import *
from hoda.tensorize import fh_power, fh_log_envelope
from hoda.classification import ZLogRatio, ZScore
dataset = AlexMI()

sfreq=250
paradigm = MotorImagery(n_classes=len(dataset.event_id), resample=sfreq)
X, y, meta = paradigm.get_data(
     dataset=dataset, 
     subjects=[1],
     return_epochs=False
)



Choosing from all possible events
/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/moabb/datasets/preprocessing.py:278: UserWarning:

warnEpochs <Epochs | 60 events (all good), 0 – 3 s (baseline off), ~11.3 MiB, data loaded,
 'right_hand': 20
 'feet': 20
 'rest': 20>



In [25]:
import matplotlib.pyplot as plt
from meeglet import define_frequencies, define_wavelets, plot_wavelet_family
import matplotlib
import numpy as np
%matplotlib inline

freqs, sigma_time, sigma_freq, bw_oct, qt = define_frequencies(
    foi_start=8, foi_end=32, bw_oct=0.5, delta_oct=1/8
)
n_cycles = 5
freqs

array([ 8.        ,  8.72406186,  9.51365692, 10.37471644, 11.3137085 ,
       12.3376866 , 13.45434264, 14.67206469, 16.        , 17.44812372,
       19.02731384, 20.74943287, 22.627417  , 24.67537321, 26.90868529,
       29.34412938, 32.        ])

In [26]:
from mne.time_frequency import tfr_array_morlet

X_tfr = tfr_array_morlet(X, sfreq, freqs, n_cycles=5, zero_mean=True, output='power', n_jobs=-1)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  13 out of  16 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  16 out of  16 | elapsed:    0.1s finished


In [27]:
target_sfreq=25
decim = int(sfreq/target_sfreq)
decim

10

In [28]:
from mne.filter import filter_data


X_tfr_base = X_tfr
X_tfr_base = np.log(X_tfr_base/np.mean(X_tfr_base, axis=(0,3))[np.newaxis,:,:,np.newaxis])
X_tfr_base /= np.std(np.log(X_tfr), axis=(0,3))[np.newaxis,:,:,np.newaxis]


for fi in range(X_tfr.shape[2]):
    X_tfr_base[:,:,fi,:] = filter_data(X_tfr_base[:,:,fi,:], sfreq, l_freq=None, h_freq=target_sfreq/2, verbose=False)

X_tfr_base = X_tfr_base[:,:,:,::decim]

In [29]:
X_tfr_base = tl.tensor(X_tfr_base)

In [30]:
from hoda.hoda import BTTDA

bttda = BTTDA(
    ranks=[None]*16,
    hoda_params=dict(
        rank=None,
        max_iter=256,
        tol=1e-6,
        shrinkage='lw',
        obj='tr',
        solver='lanczos',
        taper=False,
        extra_train_info=False,
        verbose=True,
        theta=0.5,
        refit_shrinkage=True,
        toeplitz=(2,),
       
    ),
    verbose=True,
    extra_train_info=True,

)
bttda.fit(X_tfr_base,y)

Fitting block 1/16...


Forward model :   5%|▌         | 14/255 [00:00<00:00, 269.55it/s]


Fitting block 2/16...


Forward model :  19%|█▉        | 49/255 [00:00<00:00, 277.71it/s]


Fitting block 3/16...


...

Fitting block 15/16...


Forward model :  73%|███████▎  | 186/255 [00:00<00:00, 285.42it/s]


Fitting block 16/16...


Forward model :  95%|█████████▌| 243/255 [00:00<00:00, 285.60it/s]


BTTDA(extra_train_info=True,
      hoda_params={'extra_train_info': False, 'max_iter': 256, 'obj': 'tr',
                   'rank': None, 'refit_shrinkage': True, 'shrinkage': 'lw',
                   'solver': 'lanczos', 'taper': False, 'theta': 0.5,
                   'toeplitz': (2,), 'tol': 1e-06, 'verbose': True},
      ranks=[None, None, None, None, None, None, None, None, None, None, None,
             None, None, None, None, None],
      verbose=True)

In [31]:
import pandas as pd

df = pd.DataFrame(bttda.train_info_)
df

,block,rank,mse,nmse
0,1,"(1, 2, 8)",1.101422,0.903812
1,2,"(2, 2, 9)",0.996283,0.817536
2,3,"(2, 2, 11)",0.940208,0.771522
3,4,"(2, 2, 12)",0.892496,0.732370
4,5,"(2, 2, 12)",0.870243,0.714109
5,6,"(2, 2, 13)",0.847325,0.695303
6,7,"(3, 2, 13)",0.825620,0.677493
7,8,"(3, 2, 13)",0.806408,0.661727
8,9,"(3, 2, 13)",0.788366,0.646922
9,10,"(3, 2, 13)",0.773487,0.634713


In [32]:
import seaborn as sns
sns.lineplot(data=df, x='block', y='nmse')

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/seaborn/_oldcore.py:1119: FutureWarning:

use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.

/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/seaborn/_oldcore.py:1119: FutureWarning:

use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.



<Axes: xlabel='block', ylabel='nmse'>

In [33]:
from hoda.classification import SelectFdrMin1
import numpy as np

Xt = bttda.transform(X_tfr_base)
xt = tl.to_numpy(tl.unfold(Xt,0))

select = SelectFdrMin1(alpha=.05)
xts = select.fit_transform(xt,y)

df = pd.DataFrame({
    'feature': np.arange(xt.shape[-1]),
    'F': select.scores_,
    'p_value': select.pvalues_,
    'significant': select.get_support()
})

import plotly.express as px

fig = px.bar(df, x='feature', y='F', color='feature', log_y=True)
fig.show()

In [34]:
fig = px.bar(sorted(df['F']), log_y=True)
fig.add_hline(y=1)
fig.show()

In [35]:
classes = np.unique(y)
px.imshow(np.mean(xt[y==classes[0]] , axis=0).reshape((1,-1)), zmin=-1, zmax=1, color_continuous_scale='RdBu_r')

In [36]:
px.imshow(np.mean(xt[y==classes[1]], axis=0).reshape((1,-1)), zmin=-1, zmax=1, color_continuous_scale='RdBu_r')

In [37]:

cov = np.cov(xt, rowvar=False)

px.imshow(cov, color_continuous_scale='RdBu_r', zmin=-1, zmax=1)

In [38]:
from sklearn.decomposition import PCA
import scipy.stats

pca = PCA(n_components=None, whiten=True, svd_solver='full')
xt_pca = pca.fit_transform(xt)


px.line(np.cumsum(pca.explained_variance_ratio_))

In [39]:
px.imshow(np.mean(xt_pca[y==classes[0]], axis=0)[np.newaxis,:], zmin=-6, zmax=6,  color_continuous_scale='RdBu_r')

In [40]:
px.imshow(np.mean(xt_pca[y==classes[1]], axis=0)[np.newaxis,:], zmin=-6, zmax=6,  color_continuous_scale='RdBu_r')

In [41]:

cov_pca = np.cov(xt_pca, rowvar=False)

px.imshow(cov_pca, color_continuous_scale='RdBu_r', zmin=-1, zmax=1)

In [42]:
select_pca = SelectFdrMin1(alpha=.05)
select_pca.fit_transform(xt_pca,y)

df = pd.DataFrame({
    'feature': np.arange(xt_pca.shape[-1]),
    'F': select_pca.scores_,
    'p_value': select_pca.pvalues_,
    'significant': select_pca.get_support(),
    '1min_p_value':1-select_pca.pvalues_ 
})
fig = px.bar(df, x='feature', y='F', color='feature', log_y=True)
fig.add_hline(y=1)
fig.show()

In [43]:
fig = px.bar(sorted(df['F']), log_y=True)
fig.add_hline(y=1)
fig.show()

In [44]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
x_viz = PCA(n_components=2).fit_transform(xt, y)
fig = px.scatter(x=x_viz[:,0], y=x_viz[:,1], color=y)
fig.update_layout(width=1000, height=800)